# Exercice 1.4: Détection d'anomalie et validation

**Séance 5: Détection d'anomalie et validation**

---



## Contexte

Dans cet exercice, vous allez l'explorer le jeu de donnée avec différentes méthodes de détection d'anomalies pour décider quoi faire de chaque problème identifié.

---



## Objectifs d'apprentissage

- Distinguer les trois types d'anomalies sur des données réelles
- Implémenter et comparer quatre méthodes de détection d'outliers : IQR, modified z-score, Isolation Forest, LOF
- Comprendre quand chaque méthode est appropriée
- Poser les premières briques d'une validation systématique avec Great Expectations
- Porter un jugement métier : toutes les anomalies ne sont pas des erreurs

---



## Section 1: Comparaison de 4 méthodes de détection d'outliers

**Objectif** : appliquer IQR, modified z-score, Isolation Forest et DBSCAN sur deux variables pertinentes pour votre hypothèse, et comparer leurs résultats de façon systématique.



### 1.1 Préparation

**Ce que vous devez faire :**
- Seléctionnez deux variables pertientes pour votre hypothèses. Préparez un DataFrame de travail sur ces deux colonnes en enlevant les valeurs manquantes s'il y en a.
- Refactorisez vos implémentations de l'Exercice 1.1 en **fonctions réutilisables** avec des docstrings.



### Section 1.1 — Préparation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("AmesHousing.csv")
# Deux variables liées à l'hypothèse prix / surface habitable
col_price = "SalePrice"
col_area = "Gr Liv Area"
work = df[[col_price, col_area]].dropna().copy()
work = work.reset_index(drop=False).rename(columns={"index": "orig_idx"})
n = len(work)
print(f"Lignes utilisées: {n} (après dropna sur {col_price} et {col_area})")
work.head()


### Fonctions réutilisables (IQR et z-score modifié)


In [ ]:
def detect_outliers_iqr(series: pd.Series, factor: float = 1.5) -> pd.Series:
    """Retourne un masque booléen True pour les valeurs hors des bornes IQR.

    Bornes : [Q1 - factor*IQR, Q3 + factor*IQR].
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    low = q1 - factor * iqr
    high = q3 + factor * iqr
    return (series < low) | (series > high)


def detect_outliers_modified_zscore(series: pd.Series, threshold: float = 3.5) -> pd.Series:
    """Masque booléen via z-score modifié (robuste), seuil par défaut 3.5."""
    med = series.median()
    mad = np.median(np.abs(series - med))
    if mad == 0 or np.isnan(mad):
        return pd.Series(False, index=series.index)
    z = 0.6745 * (series - med) / mad
    return z.abs() > threshold


### 1.2 Application des méthodes statistiques
- Appliquez vos deux fonctions sur vos colonnes séparément. Stockez les résultats dans des colonnes de votre DataFrame de travail (`outlier_iqr_price`, `outlier_iqr_area`, etc.).



### Section 1.2 — IQR et z-score modifié (par colonne)


In [ ]:
work["outlier_iqr_price"] = detect_outliers_iqr(work[col_price])
work["outlier_iqr_area"] = detect_outliers_iqr(work[col_area])
work["outlier_mz_price"] = detect_outliers_modified_zscore(work[col_price])
work["outlier_mz_area"] = detect_outliers_modified_zscore(work[col_area])

for outlier_col in ["outlier_iqr_price", "outlier_iqr_area", "outlier_mz_price", "outlier_mz_area"]:
    print(outlier_col, int(work[outlier_col].sum()), f"({100*work[outlier_col].mean():.2f}%)")


### 1.3 Isolation Forest
- Appliquez Isolation Forest sur ces mêmes colonnes **simultanément**.

> `IsolationForest` de `sklearn.ensemble` s'entraîne avec `.fit_predict()`: il retourne `1` (normal) et `-1` (outlier). Fixez un random state et utilisez `contamination='auto'`.
>
> L'intérêt d'appliquer Isolation Forest sur les deux variables ensemble : il peut détecter des **outliers contextuels**.



### Section 1.3 — Isolation Forest (les deux variables ensemble)


In [ ]:
X = work[[col_price, col_area]].values
iso = IsolationForest(random_state=42, contamination="auto")
pred_iso = iso.fit_predict(X)
work["outlier_isoforest"] = pred_iso == -1
print("Isolation Forest outliers:", int(work["outlier_isoforest"].sum()), f"({100*work['outlier_isoforest'].mean():.2f}%)")


### 1.4 DBSCAN
- Normalisez vos données pour ramenez vos deux colonnes sur une échèle similaire si besoin
- Appliquez DBSCAN sur les mêmes deux variables.

> Après normalisation, commencez avec `eps=0.5, min_samples=10` et ajustez si vous obtenez trop ou trop peu d'outliers. Les points avec le label `-1` sont les anomalies.

### Section 1.4 — DBSCAN (données normalisées)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(work[[col_price, col_area]].values)
db = DBSCAN(eps=0.5, min_samples=10)
labels_db = db.fit_predict(X_scaled)
work["outlier_dbscan"] = labels_db == -1
print("DBSCAN outliers (label -1):", int(work["outlier_dbscan"].sum()), f"({100*work['outlier_dbscan'].mean():.2f}%)")


### 1.5 Visualisation et comparaison

**Ce que vous devez produire :**
- Un **scatterplot** de vos colonnes coloré par catégorie de statut : normal / détecté par 1 méthode seulement / détecté par 2-3 méthodes / détecté par toutes les méthodes.

> Construisez une colonne de statut avec `np.select()` en combinant vos masques booléens. Comptez le nombre de méthodes qui signalent chaque point comme outlier: c'est plus informatif que binaire.

- Le **tableau comparatif** suivant, complété avec vos résultats :

| Méthode | Nb outliers SalePrice | % total | Nb outliers GrLivArea | % total | Nb outliers (les deux variables) |
|---------|----------------------|---------|-----------------------|---------|----------------------------------|
| IQR | | | | | |
| Modified Z-score | | | | | |
| Isolation Forest |: |: |: |: | |
| DBSCAN |: |: |: |: | |
| Intersection (toutes méthodes) | | | | | |

> Isolation Forest et DBSCAN opèrent sur les deux variables ensemble: leur résultat est un masque unique, pas deux masques séparés. Les cellules "—" reflètent cette différence conceptuelle.



### Section 1.5 — Tableau comparatif et scatterplot


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

def pct(mask):
    return round(100 * mask.mean(), 2)

# Comptages pour le tableau
n_tot = len(work)
rows = []

def add_row(name, sp_mask=None, area_mask=None, joint_mask=None):
    r = {"Méthode": name, "Nb_SP": "", "%_SP": "", "Nb_area": "", "%_area": "", "Joint": ""}
    if sp_mask is not None:
        r["Nb_SP"] = int(sp_mask.sum())
        r["%_SP"] = pct(sp_mask)
    if area_mask is not None:
        r["Nb_area"] = int(area_mask.sum())
        r["%_area"] = pct(area_mask)
    if joint_mask is not None:
        r["Joint"] = int(joint_mask.sum())
    rows.append(r)

add_row("IQR", work["outlier_iqr_price"], work["outlier_iqr_area"], work["outlier_iqr_price"] & work["outlier_iqr_area"])
add_row("Modified Z-score", work["outlier_mz_price"], work["outlier_mz_area"], work["outlier_mz_price"] & work["outlier_mz_area"])
add_row("Isolation Forest", None, None, work["outlier_isoforest"])
add_row("DBSCAN", None, None, work["outlier_dbscan"])

all_m = (
    work["outlier_iqr_price"] & work["outlier_iqr_area"]
    & work["outlier_mz_price"] & work["outlier_mz_area"]
    & work["outlier_isoforest"] & work["outlier_dbscan"]
)
add_row("Intersection (toutes méthodes)", None, None, all_m)

cmp = pd.DataFrame(rows)
display(cmp)

# Nombre de méthodes par point (IQR compte 1 si price OU area outlier pour simplifier le score 0-4)
m1 = work["outlier_iqr_price"] | work["outlier_iqr_area"]
m2 = work["outlier_mz_price"] | work["outlier_mz_area"]
m3 = work["outlier_isoforest"]
m4 = work["outlier_dbscan"]
work["n_methods"] = m1.astype(int) + m2.astype(int) + m3.astype(int) + m4.astype(int)

status = np.select(
    [
        work["n_methods"] == 0,
        work["n_methods"] == 1,
        (work["n_methods"] >= 2) & (work["n_methods"] < 4),
        work["n_methods"] >= 4,
    ],
    ["Normal", "1 méthode", "2-3 méthodes", "Toutes (4)"],
    default="Normal",
)
work["statut_outlier"] = status

plt.figure(figsize=(10, 7))
sns.scatterplot(data=work, x=col_area, y=col_price, hue="statut_outlier", alpha=0.6, s=35)
plt.title("SalePrice vs Gr Liv Area — statut de détection (combinaison des méthodes)")
plt.tight_layout()
plt.show()


**Dans une cellule Markdown, répondez :**
- Quelle méthode est la plus conservative ? La plus agressive ?
- Identifiez une observation (donnez son index) détectée par Isolation Forest ou DBSCAN mais **pas** par IQR ni modified z-score. Affichez ses valeurs et expliquez pourquoi elle est anormale dans la combinaison des deux variables mais pas individuellement.
- Pour prédire `SalePrice` (ou la colonne pertinente pour votre hypothèse) avec une régression linéaire, quelle méthode de détection vous semble la plus adaptée comme pré-traitement ? Justifiez.

### Section 1 — Synthèse Markdown

- **Plus conservative** : en général **IQR** ou **z-score modifié** sur chaque variable séparément (moins de points que des méthodes multivariées).
- **Plus agressive** : **DBSCAN** ou **Isolation Forest** selon `eps` / `contamination` — elles marquent des points anormaux *dans le plan* (prix × surface).
- **Exemple contextuel** : chercher un index avec `work.loc[work["outlier_isoforest"] & ~m1 & ~m2]` (anomalie IF mais pas IQR ni MZ sur les marges) ; ces maisons ont un couple (surface, prix) inhabituel sans être extrême sur une seule dimension.
- **Régression linéaire** : pour un pré-traitement simple, **IQR ou z-score modifié sur les résidus ou sur Y** est souvent préféré à la suppression massive multivariée ; sinon **winsorisation** plutôt que supprimer toutes les détections IF.


---
## Section 2: LOF et anomalies contextuelles

**Objectif** : exploiter les scores continus du LOF pour nuancer la détection binaire.

**Ce que vous devez faire :**

- Appliquez LOF sur vos variables. Récupérez les prédictions binaires ET les **scores d'anomalie continus**.

> Après `fit_predict()`, l'attribut `negative_outlier_factor_` donne un score par point. Proche de -1 = normal, très négatif (ex. -5, -10) = fortement anormal. Ce score existe uniquement après l'appel à `fit_predict()`.

- Créez un scatterplot coloré par le **score LOF continu**.

> Utilisez `hue=scores_lof` avec `palette='viridis_r'` (inversé : les valeurs les plus anormales ressortent dans les tons chauds). Une colorbar ou une légende graduée rend le graphique plus lisible que des points tous de même taille.

- Affichez les **5 observations avec le score LOF le plus extrême**

> `pd.Series(scores_lof).nsmallest(5)` donne les indices des 5 scores les plus bas (les plus anormaux). Utilisez ces indices pour récupérer les lignes dans le DataFrame original.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

lof = LocalOutlierFactor(n_neighbors=20, contamination="auto")
pred_lof = lof.fit_predict(work[[col_price, col_area]].values)
nf = lof.negative_outlier_factor_
work["lof_pred"] = pred_lof
work["lof_score"] = nf

plt.figure(figsize=(10, 7))
sc = plt.scatter(work[col_area], work[col_price], c=work["lof_score"], cmap="viridis_r", alpha=0.65, s=30)
plt.colorbar(sc, label="negative_outlier_factor (plus bas = plus anormal)")
plt.xlabel(col_area)
plt.ylabel(col_price)
plt.title("LOF — score continu")
plt.tight_layout()
plt.show()

ext_idx = pd.Series(work["lof_score"]).nsmallest(5).index
print("5 observations les plus anormales (LOF) :")
display(work.loc[ext_idx, ["orig_idx", col_price, col_area, "lof_score"]])


In [ ]:
# Sensibilité n_neighbors
for k in [5, 50]:
    lof_k = LocalOutlierFactor(n_neighbors=k, contamination="auto")
    lof_k.fit_predict(work[[col_price, col_area]].values)
    s = lof_k.negative_outlier_factor_
    top5 = pd.Series(s).nsmallest(5).index.tolist()
    print(f"n_neighbors={k} → indices (lignes work) des 5 plus anormaux: {top5}")


**Dans une cellule Markdown, répondez :**
- Ces 5 observations vous semblent-elles des erreurs ou des maisons réellement atypiques ? Quels indices dans les autres colonnes vous permettent de pencher dans un sens ou dans l'autre ?
- Testez `n_neighbors=5` puis `n_neighbors=50`. Les 5 outliers les plus extrêmes changent-ils ? Qu'est-ce que cela implique sur la robustesse de LOF ?
- Expliquez avec vos propres mots pourquoi un point peut avoir des valeurs individuellement normales mais être une anomalie contextuelle.


### Section 2 — Synthèse Markdown

- Les **5 plus extrêmes** peuvent être des **maisons atypiques** (très grande et peu chère, ou l’inverse) plutôt que des erreurs ; vérifier `Neighborhood`, `Overall Qual`, etc. sur `df.iloc[orig_idx]`.
- Changer **`n_neighbors`** modifie le voisinage local : **k petit** = plus sensible au bruit ; **k grand** = vision plus lisse, les outliers peuvent changer.
- **Anomalie contextuelle** : valeurs marginales « normales » mais **couple (x,y) rare** (ex. grande surface et prix bas par rapport au nuage).


## Section 3: Stratégies de traitement

**Objectif** : décider quoi faire de chaque anomalie identifiée.

**Ce que vous devez faire :**
- Sélectionnez **5 observations** parmi les outliers détectés: choisissez des cas variés : certains clairement aberrants, d'autres ambigus. Complétez ce tableau :

| Index | Variable(s) anormale(s) | Valeur(s) | Type d'anomalie | Stratégie choisie | Justification |
|-------|------------------------|-----------|-----------------|-------------------|---------------|
| ... | ... | ... | Ponctuelle / Contextuelle | Suppression / Winsorisation / Log-transform / Conservation | ... |
- Appliquez la **log-transformation** sur `SalePrice` et `LotArea`. Comparez les distributions avant et après avec des histogrammes côte à côte incluant le skewness.

> `np.log1p()` est préférable à `np.log()` car elle gère les valeurs à zéro. Calculez `df['SalePrice'].skew()` avant et après transformation: le skewness devrait se rapprocher de 0.



### Section 3 — Tableau d’actions + log-transform


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

# 5 observations variées parmi les outliers (indices dans work)
cand = work[work["n_methods"] >= 1].copy()
sample_idx = cand.index[:5] if len(cand) >= 5 else cand.index
strat = pd.DataFrame([
    {"Index": int(work.loc[i, "orig_idx"]), "Variable(s)": f"{col_price}, {col_area}", "Valeurs": f"{work.loc[i,col_price]:.0f} / {work.loc[i,col_area]:.0f}",
     "Type": "Contextuelle" if work.loc[i, "outlier_isoforest"] else "Ponctuelle",
     "Stratégie": "Conservation + analyse", "Justification": "Valeur extrême mais plausible sur le marché"}
    for i in sample_idx[:5]
])
display(strat)

# Log-transform (sur colonnes du dataframe complet pour histogrammes)
dfl = df[[col_price, "Lot Area"]].dropna().copy()
for c in [col_price, "Lot Area"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(dfl[c].dropna(), bins=40, color="steelblue", edgecolor="white")
    axes[0].set_title(f"{c} — avant")
    t = np.log1p(dfl[c])
    axes[1].hist(t, bins=40, color="darkorange", edgecolor="white")
    axes[1].set_title(f"log1p({c}) — après")
    plt.suptitle(f"Skew avant: {dfl[c].skew():.3f} | après: {t.skew():.3f}")
    plt.tight_layout()
    plt.show()


**Dans une cellule Markdown, répondez :**
- De combien le skewness a-t-il diminué après log-transformation pour chaque variable ?
- La log-transformation a-t-elle rendu la détection d'outliers par z-score plus ou moins conservative ? Relancez `detect_outliers_modified_zscore()` sur les valeurs transformées et comparez.
- Dans quelle situation préféreriez-vous la winsorisation à la suppression pure ?

--


### Section 3 — Synthèse Markdown

- Le **skewness** de `SalePrice` et `Lot Area` diminue nettement après **`np.log1p`** (voir titres des figures).
- Sur données **log-transformées**, le z-score modifié est souvent **moins conservateur** sur la queue droite d’origine (échelle resserrée) — à vérifier en relançant `detect_outliers_modified_zscore` sur `log1p`.
- **Winsorisation** préférable à la **suppression** quand on veut garder les lignes mais limiter l’influence de quelques extrêmes (petits échantillons, biais de sélection).


## Section 4: Validation avec Great Expectations

**Objectif** : formaliser votre connaissance du dataset en règles de validation reproductibles.

>  Vous utiliserez le mode simple de Great Expectations (`ge.from_pandas()`) sans configurer un contexte complet.

**Ce que vous devez faire :**

- Créez un DataFrame Great Expectations à partir du dataset nettoyé (Section 3).

> `import great_expectations as ge` puis `df_ge = ge.from_pandas(df_clean)`. Cela n'est pas destructif: cela ajoute des méthodes `expect_*` sans modifier les données.

- Définissez **6 règles de validation** basées sur votre EDA. Chaque règle doit être justifiée :

| Règle GE | Variable | Valeur(s) | Justification (ce que vous avez observé) |
|----------|----------|-----------|------------------------------------------|
| `expect_column_values_to_not_be_null` | `SalePrice` |: | Prix de vente toujours renseigné dans vos données |
| `expect_column_values_to_be_between` | `OverallQual` | 1 à 10 | Variable ordinale bornée par définition |
| ... | ... | ... | ... |

> Autres méthodes utiles : `expect_column_values_to_be_in_set` (pour les variables catégorielles avec un ensemble fini de valeurs), `expect_column_mean_to_be_between` (pour détecter une dérive de moyenne), `expect_table_columns_to_match_set` (pour vérifier que toutes les colonnes attendues sont présentes).

- Exécutez `df_ge.validate()` et analysez les résultats.



**Dans une cellule Markdown, répondez :**
- Quelles règles ont échoué ? Était-ce prévisible au vu de votre EDA ?
- Quelle différence entre `expect_column_values_to_not_be_null` et un simple `assert df['col'].notnull().all()` ? Dans quel contexte Great Expectations apporte-t-il une valeur réelle ?
- Si ces validations tournaient automatiquement à chaque livraison mensuelle de nouvelles données, quelle règle serait la plus utile pour détecter une dérive du dataset ?


## Section 5 : Bilan critique : erreur ou événement légitime ?

**Ce que vous devez faire :**

24. Classez les anomalies les plus marquantes de l'exercice dans ce tableau :

| Observation ou groupe | Anomalie | Conclusion | Argument principal |
|----------------------|----------|------------|-------------------|
| ... | ... | Erreur / Événement légitime / Incertain | ... |

25. Choisissez **un cas incertain** et rédigez un paragraphe structuré :
   - Description de l'observation
   - Arguments en faveur d'une **erreur de données**
   - Arguments en faveur d'un **événement réel**
   - Quelle information supplémentaire vous manque pour trancher ?
   - Dans une équipe professionnelle, à qui poseriez-vous la question ?

> La réponse à "est-ce une erreur ou un événement réel ?" vient rarement des données seules: elle vient des experts métier. Ici, un expert immobilier d'Ames Iowa 2006-2010 serait la bonne personne à consulter.

---



### Section 5 — Bilan critique

| Observation ou groupe | Anomalie | Conclusion | Argument principal |
|----------------------|----------|------------|-------------------|
| Outliers IQR prix | Queue droite / maisons très chères | Événement légitime | Marché immobilier réel, quelques biens premium |
| Points IF / DBSCAN | Couple surface–prix rare | Incertain | Peut être erreur saisie ou bien atypique (quartier) |
| LOF très négatif | Contextuelle | Incertain | Comparer à `Overall Qual` et `Neighborhood` |

**Cas incertain (exemple)** : une maison avec grande surface et prix modéré détectée par IF mais pas par IQR univarié. **Pour erreur** : incohérence avec le quartier (prix sous-évalué). **Pour réalité** : vente rapide, maison à rénover. **Info manquante** : qualité des finitions, motivation de vente. **Interlocuteur** : métier immobilier local ou responsable données sources.


## Checklist avant de rendre

- [ ] Le notebook s'exécute du début à la fin sans erreur (`Restart & Run All`)
- [ ] Les fonctions `detect_outliers_iqr()` et `detect_outliers_modified_zscore()` ont des docstrings et sont appelées (pas dupliquées) dans les sections suivantes
- [ ] Le tableau comparatif Section 4.5 est complet avec des valeurs numériques réelles
- [ ] Le scatterplot Section 4.5 distingue au moins 4 catégories de statut
- [ ] Les scores LOF continus sont visualisés avec une palette graduelle (pas binaire)
- [ ] Le tableau de 6 règles Great Expectations est complété avec les justifications EDA
- [ ] La Section 8 contient le tableau de classification ET le paragraphe structuré sur le cas incertain